# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Group 24"
student_name = "Mukesh Murugesan"
student_id = "25747763"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# Install scikit-learn for modelling utilities
!pip install scikit-learn

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THIS LINE:
import pandas as pd
import altair as alt

# ADD these below:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

In [ ]:
# ── BUSINESS CONTEXT & HYPOTHESIS ───────────────────────────────
# Use Case (Student B — Regression):
#   Predict the total sales amount (line_total) for each order line item.
#
# Business Problem:
#   The retailer has no way to estimate revenue per order line before fulfilment.
#   A regression model enables real-time revenue forecasting, smarter inventory
#   planning, and dynamic pricing decisions.
#
# Hypothesis:
#   Product category, order quantity, pricing position (price_vs_list), and
#   discount status are strong predictors of line_total. A Random Forest
#   regression model trained on these features will achieve R2 >= 0.80 on
#   the validation set — outperforming the mean baseline significantly.

print("Business context defined.")
print("Target: line_total | Method: Regression | Expected R2 >= 0.80")

### A.1 Approach 1

In [ ]:
# ── APPROACH 1: JOIN TABLES + DOMAIN KNOWLEDGE ──────────────────
# line_total is driven by product type, quantity, discount and customer context.

# Step 1: sales_order_detail + sales_order_header (get order date, channel)
merged = sales_order_detail_df.merge(
    sales_order_header_df[['sales_order_id','customer_id','order_date','online_order_flag']],
    on='sales_order_id', how='left'
)
print("After +order header:", merged.shape)

# Step 2: + customer (get person_id)
merged = merged.merge(
    customer_df[['customer_id','person_id']].drop_duplicates('customer_id'),
    on='customer_id', how='left'
)
print("After +customer:", merged.shape)

# Step 3: + person (get email_promotion)
person_clean = person_df[['person_id','email_promotion']].drop_duplicates('person_id')
merged = merged.merge(person_clean, on='person_id', how='left')
print("After +person:", merged.shape)

# Step 4: + product (get list_price, standard_cost, product_line)
merged = merged.merge(
    product_df[['product_id','product_subcategory_id','list_price','standard_cost','product_line']].drop_duplicates('product_id'),
    on='product_id', how='left'
)
print("After +product:", merged.shape)

# Step 5: + product_sub_category + product_category (get category_name)
merged = merged.merge(
    product_sub_category_df[['product_subcategory_id','product_category_id']],
    on='product_subcategory_id', how='left'
)
merged = merged.merge(
    product_category_df[['product_category_id','name']].rename(columns={'name':'category_name'}),
    on='product_category_id', how='left'
)
print("After +category:", merged.shape)
print()
print("All columns available:")
print(merged.columns.tolist())

In [ ]:
feature_selection_1_insights = """
Approach 1: Domain Knowledge Feature Selection via Table Joins

Six tables were joined to the core sales_order_detail dataset to enrich
each line item with contextual information.

Join chain:
  sales_order_detail → sales_order_header → customer → person
  sales_order_detail → product → product_sub_category → product_category

Candidate features identified through domain knowledge:

From sales_order_detail (core):
  - order_quantity:        directly multiplies into line_total
  - unit_price_discount:   directly reduces line_total
  - has_discount:          binary flag — discounted orders behave differently

From sales_order_header:
  - order_date (→ year/month): captures seasonal and growth trends
  - online_order_flag:     online vs in-store orders differ in average value

From person:
  - email_promotion:       customer marketing engagement level

From product + category:
  - category_name:         Bikes have high unit prices; Accessories are low
  - product_line:          R(Road), M(Mountain), T(Touring) differ in price tier
  - price_vs_list:         engineered ratio of unit_price to list_price

Features EXCLUDED to prevent data leakage:
  - unit_price:  directly in the formula line_total = qty x unit_price x (1-disc).
                 Using it would allow the model to simply recompute the answer
                 rather than learn a generalisable pattern. Replaced by price_vs_list.
  - sub_total, total_due, tax_amount: computed FROM line_total — leakage.
  - All ID columns: no predictive signal, cause memorisation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2

In [ ]:
# ── APPROACH 2: CORRELATION ANALYSIS ────────────────────────────
# Encode categoricals temporarily for correlation computation

df_corr = merged[['order_quantity','unit_price_discount','online_order_flag',
                   'email_promotion','line_total']].copy()

corr_result = df_corr.corr()['line_total'].drop('line_total').sort_values(ascending=False)
print("Pearson Correlation with line_total:")
print(corr_result.round(4))
print()

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if v >= 0 else 'tomato' for v in corr_result.values]
ax.barh(corr_result.index, corr_result.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson Correlation of Numeric Features with line_total', fontsize=12)
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
feature_selection_2_insights = """
Approach 2: Pearson Correlation-Based Feature Selection

Correlation coefficients were computed between numeric features and line_total
to statistically validate the domain knowledge selections from Approach 1.

Key findings:
  - order_quantity:      r = 0.248  — moderate positive; confirmed as useful feature
  - unit_price_discount: r = 0.045  — weak linear correlation BUT EDA showed strong
                                       non-linear pattern (discounted orders avg $2,538
                                       vs $706 non-discounted) — kept despite weak r
  - online_order_flag:   r = -0.197 — negative; online orders tend to be lower value
                                       than in-store orders
  - email_promotion:     r = 0.004  — very weak; kept for customer context

Limitation of Pearson correlation:
  Pearson only detects LINEAR relationships. unit_price_discount showed a
  non-linear pattern in EDA that correlation analysis would incorrectly dismiss.
  This confirms domain knowledge (Approach 1) must complement statistical analysis.

Decision: all candidate features from Approach 1 are retained.
Correlation analysis confirms no features need to be dropped.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.n Approach "\<DATA LEAKAGE CHECK\>"

> You can add more cells related to other approaches in this section

In [ ]:
# ── APPROACH 3: DATA LEAKAGE CHECK ──────────────────────────────
# Identify any features that would not be available BEFORE an order is placed,
# or features mathematically derived from line_total

leakage_features = ['unit_price', 'sub_total', 'tax_amount', 'freight', 'total_due']
print("Leakage check — features that MUST be excluded:")
print()
for feat in leakage_features:
    if feat in merged.columns:
        corr_val = merged[[feat,'line_total']].corr()['line_total'][feat]
        print(f"  {feat}: r={corr_val:.4f} ← EXCLUDED (leakage)")
    else:
        print(f"  {feat}: not in dataset")

print()
print("Reason: unit_price is literally IN the formula for line_total.")
print("sub_total/total_due are aggregated from line_total — direct leakage.")
print("A model using these would score near 100% but fail on real new orders.")

In [ ]:
feature_selection_n_insights = """
Approach 3: Data Leakage Prevention

Data leakage occurs when features are used that would not be available
at prediction time, or that are derived from the target variable.

Leakage features identified and removed:
  - unit_price (r=0.734 with line_total):
    Although strongly correlated, unit_price is part of the mathematical formula:
    line_total = order_quantity x unit_price x (1 - discount).
    Including it lets the model recompute the answer exactly — this is not a
    generalisable prediction, it is arithmetic. Replaced by price_vs_list ratio.

  - sub_total, tax_amount, freight, total_due (from sales_order_header):
    These are ORDER-level aggregates computed from line_total itself.
    Using them would create circular reasoning.

Impact of removing unit_price:
  - Model R2 drops from ~1.00 to ~0.87 on the validation set.
  - This is the honest, deployable model performance.
  - A model that generalises to unseen orders must learn from product category,
    pricing tier (price_vs_list), and order context — not the exact unit price.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
# ── FINAL FEATURE LIST ───────────────────────────────────────────
# Defined here — used through the rest of the notebook

features_list = [
    'order_quantity',       # units ordered — key quantity driver
    'unit_price_discount',  # discount applied (0.0 to 0.4)
    'has_discount',         # binary: 1 if any discount applied
    'online_order_flag',    # 1=online, 0=in-store
    'order_year',           # extracted from order_date
    'order_month',          # extracted from order_date
    'email_promotion',      # customer preference (0, 1, 2)
    'category_name',        # Bikes/Accessories/Clothing/Components
    'product_line',         # R/M/T/S/Unknown
    'price_vs_list',        # unit_price / list_price ratio
]

print("Final features selected:", len(features_list))
for i, f in enumerate(features_list, 1):
    print(f"  {i:2d}. {f}")
print()
print("Target variable: line_total (log1p transformed in Section E)")

In [ ]:
feature_selection_explanations = """
10 features selected across 3 selection approaches:

  1. order_quantity     — core quantity driver (r=0.248 with line_total)
  2. unit_price_discount — discount proportion; non-linear effect confirmed in EDA
  3. has_discount       — binary flag; cleaner signal for the 2% discounted orders
  4. online_order_flag  — channel difference: online orders average lower value
  5. order_year         — captures year-over-year business growth (2011-2014)
  6. order_month        — captures seasonal purchasing patterns
  7. email_promotion    — customer engagement level (0=none, 1=AW, 2=partner)
  8. category_name      — most important category signal: Bikes are high value,
                          Accessories are low value — 4 categories total
  9. product_line       — R(Road)/M(Mountain)/T(Touring)/S(Standard)/Unknown
  10. price_vs_list     — unit_price / list_price: captures pricing position
                          without the leakage risk of using raw unit_price

unit_price excluded: direct formula component — causes data leakage (R2 near 1.0).
All ID columns excluded: no predictive signal.
standard_cost excluded: 4,431 missing values (10.5%) — too risky.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing "\<Missing values in email_promotion, product_line, and online_order_flag.
\>"

In [ ]:
# ── B.1: FIX MISSING VALUES ─────────────────────────────────────
print("Missing values BEFORE cleaning:")
check_cols = features_list + ['line_total']
# order_year/order_month/has_discount/price_vs_list not yet created — check source cols
source_check = ['order_quantity','unit_price_discount','online_order_flag',
                'email_promotion','category_name','product_line','line_total']
print(merged[source_check].isnull().sum())
print()

# Fix 1: email_promotion — fill with mode (0 = no promotion)
merged['email_promotion'] = merged['email_promotion'].fillna(0).astype(int)

# Fix 2: product_line — fill with 'Unknown' (no safe assumption)
merged['product_line'] = merged['product_line'].fillna('Unknown')

# Fix 3: online_order_flag — fill with mode then convert to int
mode_flag = merged['online_order_flag'].mode()[0]
merged['online_order_flag'] = merged['online_order_flag'].fillna(mode_flag).astype(int)

print("Missing values AFTER cleaning:")
print(merged[source_check].isnull().sum())
print()
print("All missing values resolved.")

In [ ]:
data_cleaning_1_explanations = """
Issue: Missing values in email_promotion, product_line, and online_order_flag.

Root causes and fixes:

  email_promotion (291 nulls — 0.69%):
    Some person records have no marketing preference recorded.
    Fix: impute with mode = 0 (no promotion). Justified because 55.4% of all
    customers have this value — it is the safest conservative assumption.

  product_line (665 nulls — 1.58%):
    Some products have no product line assigned in the product table.
    Fix: impute with 'Unknown' category. We cannot safely guess whether
    a missing product line is Road, Mountain or Touring.

  online_order_flag (small number of nulls):
    A few orders have no channel recorded.
    Fix: impute with mode = 1 (online). The majority of orders are online.

Why not drop rows?
    Dropping 665 rows would remove 1.58% of training data with no benefit.
    More importantly, these rows are not missing the TARGET variable (line_total
    has zero nulls), so they carry valid training signal.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "\<Incorrect data types preventing feature engineering and modelling\>"

In [ ]:
# ── B.2: FIX DATA TYPES ─────────────────────────────────────────
print("Data types BEFORE fixing:")
print(merged[['order_date','online_order_flag','email_promotion']].dtypes)
print()

# Convert order_date string to datetime
merged['order_date'] = pd.to_datetime(merged['order_date'])

# Ensure integer types for flag columns
merged['online_order_flag'] = merged['online_order_flag'].astype(int)
merged['email_promotion']   = merged['email_promotion'].astype(int)

print("Data types AFTER fixing:")
print(merged[['order_date','online_order_flag','email_promotion']].dtypes)
print()
print("order_date now datetime64 — ready for year/month extraction in Section D.")

In [ ]:
data_cleaning_2_explanations = """
Issue: Incorrect data types preventing feature engineering and modelling.

Problems and fixes:
  - order_date loaded as object (string) — must be datetime to extract
    order_year and order_month in Section D.
    Fix: pd.to_datetime(merged['order_date'])

  - online_order_flag stored as float64 (due to NaN values during loading).
    It should be a binary integer (0 or 1).
    Fix: cast to int after imputation.

  - email_promotion stored as float64 for the same reason.
    Fix: cast to int — values are 0, 1, or 2.

Impact: Without these fixes, extracting order_year with .dt.year would raise
an AttributeError. Incorrect float types would also produce unexpected behaviour
during label encoding and model training.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "\<REMOVE IRRELEVANT AND LEAKAGE COLUMNS\>"

In [ ]:
# ── B.3: REMOVE IRRELEVANT AND LEAKAGE COLUMNS ──────────────────
cols_to_drop = [
    'sales_order_id',        'sales_order_detail_id',
    'product_id',            'special_offer_id',
    'customer_id',           'person_id',
    'product_subcategory_id','product_category_id',
    'unit_price',            # DATA LEAKAGE — in formula for line_total
    'list_price',            # kept only for price_vs_list ratio (Section D)
    'standard_cost',         # 10.5% missing — too risky
    'order_date',            # replaced by order_year and order_month (Section D)
]

cols_to_drop = [c for c in cols_to_drop if c in merged.columns]
merged_clean = merged.drop(columns=cols_to_drop)

print("Columns before removal:", merged.shape[1])
print("Columns after removal: ", merged_clean.shape[1])
print()
print("Remaining columns:")
print(merged_clean.columns.tolist())

In [ ]:
data_cleaning_3_explanations = """
Issue: Dataset contains ID columns, leakage columns, and redundant columns.

Removed categories:

  ID columns (sales_order_id, sales_order_detail_id, customer_id, person_id,
  product_id, special_offer_id, product_subcategory_id, product_category_id):
    UUIDs have no predictive signal. Including them causes the model to memorise
    specific transactions rather than learning generalisable patterns.

  unit_price (DATA LEAKAGE):
    unit_price is directly in the formula: line_total = qty x unit_price x (1-disc).
    A model using unit_price achieves R2=1.00 by recomputing the arithmetic — this
    is not generalisation. Replaced by price_vs_list (the ratio to list price),
    which captures pricing position without leakage.

  standard_cost (10.5% missing):
    4,431 null values. Imputing such a high proportion risks introducing systematic
    bias. Dropped in favour of price_vs_list which captures similar price-tier signal.

  order_date (redundant after engineering):
    Replaced by order_year and order_month in Section D.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.n Fixing "\<Final Data Quality Verification\>"

> You can add more cells related to other issues in this section

In [ ]:
# ── B.n: FINAL DATA QUALITY CHECK ───────────────────────────────
print("Final dataset shape:", merged_clean.shape)
print()
print("Remaining missing values:")
remaining_nulls = merged_clean.isnull().sum()
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.sum() > 0 else "None — dataset is clean.")
print()
print("Duplicate rows:", merged_clean.duplicated().sum())
print()
print("Sample of clean dataset:")
merged_clean.head(3)

In [ ]:
data_cleaning_n_explanations = """
Final Data Quality Verification

After all cleaning steps:
  - Shape: 42,100 rows x remaining columns (no rows dropped — all issues resolved
    through imputation and column removal).
  - Missing values: zero across all columns retained for modelling.
  - Duplicates: zero — confirmed clean.

Summary of cleaning steps:
  B.1: Imputed missing values in email_promotion, product_line, online_order_flag.
  B.2: Fixed data types for order_date, online_order_flag, email_promotion.
  B.3: Removed 12 columns — ID fields, leakage features, high-missing columns.

The dataset is now clean and ready for the train/validation/test split in Section C.
Zero rows were deleted, preserving all 42,100 training examples.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Split Datasets


In [ ]:
# ── C: SPLIT DATASETS — 70% / 15% / 15% ────────────────────────
from sklearn.model_selection import train_test_split

# Define working columns (before feature engineering — year/month/has_discount added in D)
working_cols = ['order_quantity','unit_price_discount','online_order_flag',
                'email_promotion','category_name','product_line','line_total']

df_split = merged_clean[working_cols].dropna().copy()
print("Dataset for splitting:", df_split.shape)
print()

# Step 1: split off 30% temp (val + test)
train_raw, temp_raw = train_test_split(df_split, test_size=0.30, random_state=42)

# Step 2: split temp into 50/50 → each is 15% of total
val_raw, test_raw = train_test_split(temp_raw, test_size=0.50, random_state=42)

# Assign the raw dataframes to the training, validation, and testing variables
training_df   = train_raw.copy()
validation_df = val_raw.copy()
testing_df    = test_raw.copy()

print(f"Training set:   {training_df.shape}   → {len(training_df)/len(df_split)*100:.0f}% of data")
print(f"Validation set: {validation_df.shape}   → {len(validation_df)/len(df_split)*100:.0f}% of data")
print(f"Testing set:    {testing_df.shape}   → {len(testing_df)/len(df_split)*100:.0f}% of data")
print()
print("Target (line_total) distribution across splits:")
for name, df_ in [('Train', training_df), ('Val', validation_df), ('Test', testing_df)]:
    print(f"  {name}: mean=${df_['line_total'].mean():.2f}  median=${df_['line_total'].median():.2f}")

In [ ]:
# Add these lines at the bottom of Cell 57 (after the split code)
training_df   = train_raw.copy()
validation_df = val_raw.copy()
testing_df    = test_raw.copy()

print(f"Training set:   {training_df.shape}   → {len(training_df)/len(df_split)*100:.0f}%")
print(f"Validation set: {validation_df.shape}   → {len(validation_df)/len(df_split)*100:.0f}%")
print(f"Testing set:    {testing_df.shape}   → {len(testing_df)/len(df_split)*100:.0f}%")

In [ ]:
data_splitting_explanations = """
Data Splitting Strategy: 70% Train / 15% Validation / 15% Test

Rationale for 70/15/15:
  With 42,100 rows, this gives:
    Training set:   ~29,470 rows — sufficient data for all models including
                    tree ensembles which need more examples.
    Validation set: ~6,315 rows — large enough for stable metric estimates
                    during hyperparameter tuning experiments.
    Test set:       ~6,315 rows — held out entirely until final evaluation,
                    providing an unbiased estimate of real-world performance.

Why random split (not time-based):
  The regression target is line_total for a SINGLE line item — this is not
  a time-series forecasting problem. The value of a line item does not depend
  on past line items. A random split ensures each split has the same
  distribution of product categories and order sizes.

Data leakage prevention:
  All feature engineering (Section D) and all transformations (Section E) are
  applied AFTER the split. Encoders and scalers are fit ONLY on the training
  set and applied to validation and test sets without re-fitting.
  This prevents any information from validation/test data influencing the model.

Random state = 42 used throughout for full reproducibility.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature "\<has_discount (binary flag)\>"



In [ ]:
# ── D.1: has_discount — binary discount flag ─────────────────────
# EDA: 97.99% of orders have no discount. Discounted orders average
# $2,538 vs $706 for non-discounted. A binary flag captures this split cleanly.

for df_ in [training_df_eng, validation_df_eng, testing_df_eng]:
    df_['has_discount'] = (df_['unit_price_discount'] > 0).astype(int)

print("has_discount distribution (training set):")
print(training_df_eng['has_discount'].value_counts())
print()
print(f"Discounted rows:     {training_df_eng['has_discount'].sum()}")
print(f"Non-discounted rows: {(training_df_eng['has_discount']==0).sum()}")

In [ ]:
feature_engineering_1_explanations = """
New Feature: has_discount (binary flag)
Values: 1 if unit_price_discount > 0, else 0.

Motivation from EDA:
  - 97.99% of line items have NO discount (41,253 of 42,100 rows).
  - Despite weak Pearson r (0.045), EDA revealed a strong non-linear pattern:
    orders with 20% discount average $2,538 vs $706 for non-discounted orders.
  - The raw unit_price_discount value is nearly always 0.0, making it hard
    for linear models to detect the presence of a promotional discount.

Impact:
  - has_discount gives the model a clean binary signal to distinguish whether
    a discount is present, complementing the magnitude in unit_price_discount.
  - Applied consistently across training, validation, and test sets.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature "\<order_year (2011-2014) and order_month (1-12)\>"



In [ ]:
# ── D.2: order_year and order_month ─────────────────────────────
# Extract temporal features from the original merged dataset using index

date_series = pd.to_datetime(merged_clean.loc[:, 'order_date']
                              if 'order_date' in merged_clean.columns
                              else merged['order_date'])

for df_ in [training_df_eng, validation_df_eng, testing_df_eng]:
    df_['order_year']  = merged['order_date'].iloc[df_.index if hasattr(df_.index, '__iter__') else range(len(df_))].values if False else pd.to_datetime(merged.loc[df_.index, 'order_date']).dt.year.values
    df_['order_month'] = pd.to_datetime(merged.loc[df_.index, 'order_date']).dt.month.values

print("order_year distribution (training):")
print(training_df_eng['order_year'].value_counts().sort_index())
print()
print("order_month distribution (training):")
print(training_df_eng['order_month'].value_counts().sort_index())

In [ ]:
feature_engineering_2_explanations = """
New Features: order_year (2011-2014) and order_month (1-12)

Motivation:
  The raw order_date string cannot be used in regression models.
  Extracting year and month captures two types of temporal signal:

  order_year:
    The business grew rapidly from 2011 to 2014:
      2011: 735 orders | 2012: 3,297 | 2013: 21,512 | 2014: 16,556
    As the business scaled, product range and pricing evolved.
    Year captures these long-term structural changes.

  order_month:
    Orders are distributed relatively evenly across months but show mild
    seasonal patterns (March and April are slightly higher volume months).
    Month allows tree-based models to detect seasonal pricing effects.

Both are treated as numeric ordinal features (no cyclical encoding applied
at this stage — a potential improvement for future experimentation).
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 New Feature "\<price_vs_list (unit_price / list_price)
\>"



In [ ]:
# ── D.3: price_vs_list — pricing position ratio ─────────────────
# Ratio of unit_price to the product's standard list_price.
# Captures whether the product was sold at, above, or below list price.
# This is a SAFE alternative to unit_price that avoids data leakage.

list_price_series = merged.loc[:, 'list_price']

for df_ in [training_df_eng, validation_df_eng, testing_df_eng]:
    lp = list_price_series.reindex(df_.index)
    up = merged.loc[df_.index, 'unit_price']
    df_['price_vs_list'] = (up / lp.replace(0, np.nan)).fillna(1.0).round(4)

print("price_vs_list statistics (training):")
print(training_df_eng['price_vs_list'].describe().round(4))
print()
print("Values below 1.0 (sold below list price):",
      (training_df_eng['price_vs_list'] < 1.0).sum())
print("Values equal to 1.0 (sold at list price):",
      (training_df_eng['price_vs_list'] == 1.0).sum())

In [ ]:
feature_engineering_3_explanations = """
New Feature: price_vs_list (unit_price / list_price)

Motivation:
  unit_price cannot be used directly — it causes data leakage because it is
  part of the exact formula for line_total. However, we still need to capture
  the pricing tier of the product.

  price_vs_list = unit_price / list_price solves this:
    Value = 1.0  → product sold at standard list price (most common)
    Value < 1.0  → product sold below list (promotional discount)
    Value > 1.0  → product sold above list (currency or surcharge adjustment)

Business relevance:
  Products with high list prices (premium bikes) will have different price_vs_list
  distributions than low-cost accessories. This feature encodes the RELATIVE
  pricing position rather than the absolute price, providing meaningful signal
  while avoiding the leakage issue.

Missing value handling:
  Where list_price is zero or missing, price_vs_list is set to 1.0 (neutral).
  Applied consistently across all three splits.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.n Fixing "\<FINAL FEATURE ENGINEERING VERIFICATION \>"

> You can add more cells related to new features in this section

In [ ]:
# ── D.n: FINAL FEATURE ENGINEERING VERIFICATION ─────────────────
# Confirm all 4 engineered features are present and correct in all 3 splits

engineered_features = ['has_discount', 'order_year', 'order_month', 'price_vs_list']

print("=== ENGINEERED FEATURES SUMMARY ===")
print()

for split_name, df_ in [('training_df_eng', training_df_eng),
                         ('validation_df_eng', validation_df_eng),
                         ('testing_df_eng', testing_df_eng)]:
    print(f"{split_name} — shape: {df_.shape}")
    print(f"  Columns: {df_.columns.tolist()}")
    print(f"  Nulls:   {df_.isnull().sum().sum()}")
    print()

print("=== DISTRIBUTION CHECK (training set) ===")
print()
print("has_discount:  ", training_df_eng['has_discount'].value_counts().to_dict())
print("order_year:    ", training_df_eng['order_year'].value_counts().sort_index().to_dict())
print("order_month:   min={}, max={}".format(
    training_df_eng['order_month'].min(), training_df_eng['order_month'].max()))
print("price_vs_list: min={:.2f}, max={:.2f}, mean={:.4f}".format(
    training_df_eng['price_vs_list'].min(),
    training_df_eng['price_vs_list'].max(),
    training_df_eng['price_vs_list'].mean()))

In [ ]:
feature_engineering_n_explanations = """
Feature Engineering Summary — All 4 new features verified.

Features added in Section D:

  D.1 has_discount (binary):
    Created from unit_price_discount > 0.
    Training distribution: {0: 28,884, 1: 586}
    97.99% of orders have no discount — binary flag separates these cleanly.

  D.2 order_year (2011-2014) and order_month (1-12):
    Extracted from order_date using index alignment with the original merged table.
    Training year distribution: 2011=507, 2012=2,278, 2013=15,085, 2014=11,600.
    Months distributed evenly (2,200-3,000 per month) — mild seasonal variation.

  D.3 price_vs_list (ratio: unit_price / list_price):
    Range: 0.20 to 1.00. Mean: 0.89 (most products sold at or below list price).
    50th percentile = 1.0 (majority sold at exact list price).
    Products sold below list (value < 1.0) represent promotional pricing events.

All features verified across all 3 splits:
  training_df_eng:   29,470 rows x 11 columns — 0 nulls
  validation_df_eng:  6,315 rows x 11 columns — 0 nulls
  testing_df_eng:     6,315 rows x 11 columns — 0 nulls

All feature engineering applied AFTER the train/val/test split to prevent
any information from leaking between splits.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation <put_name_here>


In [ ]:
# ── E.1: EXTRACT TARGET AND LOG1P TRANSFORM ─────────────────────
# Separate y (target) from X (features)
# Apply log1p to fix right-skewed target (skewness = 3.91 confirmed in EDA)

final_features = [
    'order_quantity', 'unit_price_discount', 'has_discount',
    'online_order_flag', 'order_year', 'order_month',
    'email_promotion', 'category_name', 'product_line', 'price_vs_list'
]

# Extract raw target BEFORE modifying X
y_train_raw = X_train['line_total'].copy()
y_val_raw   = X_val['line_total'].copy()
y_test_raw  = X_test['line_total'].copy()

# Apply log1p transformation
y_train = np.log1p(y_train_raw)
y_val   = np.log1p(y_val_raw)
y_test  = np.log1p(y_test_raw)

# Keep only feature columns
X_train = X_train[final_features].copy()
X_val   = X_val[final_features].copy()
X_test  = X_test[final_features].copy()

print("Target transformation:")
print(f"  Before log1p — skewness: {y_train_raw.skew():.3f}  (strongly right-skewed)")
print(f"  After  log1p — skewness: {y_train.skew():.3f}   (approximately normal)")
print()
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

In [ ]:
data_transformation_1_explanations = """
Transformation: log1p applied to target variable line_total.

Justification:
  EDA confirmed line_total has skewness = 3.91 and kurtosis = 28.99.
  This violates the normality assumption of OLS linear regression.
  After log1p transformation, skewness drops from 3.91 to 0.25 —
  approximately normal and suitable for linear models.

Formula: y_transformed = log(line_total + 1)
  The +1 guard prevents log(0) errors (no zero values exist, but
  it is best practice for robustness).

Back-transformation at evaluation time:
  y_predicted_usd = expm1(y_predicted_log) = e^y_predicted - 1
  All final metrics (RMSE, MAE) are reported in USD using expm1()
  so results remain meaningful for business stakeholders.

Applied to: y_train, y_val, y_test — all three splits consistently.
X features are NOT modified in this step.

Reference: James et al. (2021), An Introduction to Statistical Learning,
Chapter 3 — log transformation is standard practice for right-skewed targets.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation <put_name_here>

In [ ]:
# ── E.2: LABEL ENCODE CATEGORICAL FEATURES ──────────────────────
# category_name and product_line are strings
# sklearn models require all-numeric input

from sklearn.preprocessing import LabelEncoder

# One encoder per column — fit on TRAINING SET ONLY
le_category = LabelEncoder()
le_product  = LabelEncoder()

X_train['category_name'] = le_category.fit_transform(X_train['category_name'].astype(str))
X_val['category_name']   = le_category.transform(X_val['category_name'].astype(str))
X_test['category_name']  = le_category.transform(X_test['category_name'].astype(str))

X_train['product_line']  = le_product.fit_transform(X_train['product_line'].astype(str))
X_val['product_line']    = le_product.transform(X_val['product_line'].astype(str))
X_test['product_line']   = le_product.transform(X_test['product_line'].astype(str))

print("category_name classes:", list(le_category.classes_))
print("  → Accessories=0, Bikes=1, Clothing=2, Components=3")
print()
print("product_line classes: ", list(le_product.classes_))
print("  → M=0(Mountain), R=1(Road), S=2(Standard), T=3(Touring), Unknown=4")
print()
print("X_train dtypes after encoding:")
print(X_train.dtypes)

In [ ]:
data_transformation_2_explanations = """
Transformation: Label Encoding for category_name and product_line.

Columns encoded:
  - category_name: 4 classes → Accessories=0, Bikes=1, Clothing=2, Components=3
  - product_line:  5 classes → M=0, R=1, S=2, T=3, Unknown=4

Why label encoding:
  sklearn regression models require numeric inputs. Label encoding converts
  string categories to integers efficiently.

  For tree-based models (Random Forest, Gradient Boosting) label encoding
  is appropriate — trees make binary split decisions (category < 2 or >= 2)
  and do not assume any numeric ordering between categories.

  For linear regression, one-hot encoding would be preferable.
  Both approaches will be compared in the regression notebook (Section C).

Critical anti-leakage practice:
  LabelEncoder is FIT only on X_train — it learns the class mapping from
  training data only. The same fitted encoder is applied (transform only)
  to X_val and X_test. This ensures the model never sees the category
  distributions from validation or test data during training.

Business relevance of category_name:
  This is the single most important feature — RF feature importance = 0.878.
  Bikes have high unit prices (avg $1,000+), Accessories are low (<$50).
  The category captures the pricing tier of the entire product more cleanly
  than any other available feature.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Data Transformation <put_name_here>


In [ ]:
# ── E.3: STANDARD SCALING ON CONTINUOUS FEATURES ────────────────
from sklearn.preprocessing import StandardScaler

# Only scale continuous numeric features
# Binary and encoded categoricals do NOT need scaling
scale_cols = ['order_quantity', 'unit_price_discount', 'price_vs_list']

scaler = StandardScaler()

# Fit on TRAINING SET ONLY — apply to all 3
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols]   = scaler.transform(X_val[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print("Scaling applied to:", scale_cols)
print()
print("Training set statistics after scaling:")
print(X_train[scale_cols].describe().round(3))
print()
print("Verification — mean should be ~0.0 and std ~1.0 for training set.")

In [ ]:
data_transformation_3_explanations = """
Transformation: StandardScaler on continuous numeric features.

Features scaled: order_quantity, unit_price_discount, price_vs_list.

Formula: z = (x - mean_train) / std_train
  Each feature centred at 0 with standard deviation of 1.

Why scaling is necessary:
  - order_quantity ranges 1 to 36.
  - unit_price_discount ranges 0.0 to 0.40.
  - price_vs_list ranges 0.20 to 1.00.
  Without scaling, regularised linear models (Ridge, Lasso) penalise
  coefficients proportional to feature magnitude rather than importance.
  A feature ranging 1-36 gets a different penalty than one ranging 0-0.4,
  distorting the model even if both are equally important.

Features NOT scaled (already on appropriate scales):
  - has_discount, online_order_flag: binary 0/1 — no scaling needed.
  - order_year, order_month, email_promotion: small integer ranges.
  - category_name, product_line: encoded integers representing categories,
    not true magnitudes — scaling would be misleading.

Anti-leakage: StandardScaler is fit on X_train mean and std ONLY.
The SAME fitted scaler transforms X_val and X_test — they never contribute
to the scaling parameters.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.n Fixing "\<Final Verification Summary\>"

> You can add more cells related to data preparation in this section

In [ ]:
# ── E.n: FINAL VERIFICATION ─────────────────────────────────────
print("=" * 50)
print("FINAL DATASET VERIFICATION")
print("=" * 50)
print()
print("--- SHAPES ---")
print(f"  X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}    |  y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")
print()
print("--- MISSING VALUES ---")
print(f"  X_train nulls: {X_train.isnull().sum().sum()}")
print(f"  X_val nulls:   {X_val.isnull().sum().sum()}")
print(f"  X_test nulls:  {X_test.isnull().sum().sum()}")
print()
print("--- TARGET DISTRIBUTION (log scale) ---")
for name, y_ in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    print(f"  {name}: mean={y_.mean().item():.3f}  std={y_.std().item():.3f}  skew={y_.skew().item():.3f}")
print()
print("--- FEATURE LIST ---")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col}")
print()
print("All checks passed. Ready to save.")

In [ ]:
data_transformation_n_explanations = """
Final Verification Summary — All preparation steps complete.

Shapes confirmed:
  X_train: 29,470 rows x 10 features
  X_val:    6,315 rows x 10 features
  X_test:   6,315 rows x 10 features

Data quality:
  Zero missing values across all 6 arrays (X and y for all 3 splits).
  Target skewness reduced from 3.91 to 0.25 after log1p transformation.

Anti-leakage confirmation (critical for HD Criterion 4):
  - train_test_split performed BEFORE all feature transformations.
  - LabelEncoder fit on X_train only — same encoder applied to X_val/X_test.
  - StandardScaler fit on X_train only — same scaler applied to X_val/X_test.
  - unit_price excluded and replaced by price_vs_list to prevent formula leakage.

Full pipeline summary:
  A: 6-table join, domain knowledge, correlation analysis, leakage check
     → 10 candidate features selected
  B: Imputed nulls (email_promotion, product_line, online_order_flag),
     fixed data types, removed 12 ID/leakage columns
  C: 70/15/15 random split (random_state=42 for reproducibility)
  D: Engineered has_discount, order_year, order_month, price_vs_list
  E: log1p target, LabelEncoder (train-fit only), StandardScaler (train-fit only)

Expected model performance on validation set (confirmed by pipeline testing):
  Baseline (mean predictor):   R2=0.00  USD_RMSE=$1,464  USD_MAE=$710
  Linear Regression:           R2=0.34  USD_RMSE=$1,435  USD_MAE=$696
  Random Forest (100 trees):   R2=0.87  USD_RMSE=$633    USD_MAE=$262
  Gradient Boosting (100):     R2=0.88  USD_RMSE=$667    USD_MAE=$298
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

In [ ]:
# ── CONVERT y TO DATAFRAMES BEFORE SAVING ───────────────────────
# The save cell expects DataFrames, not Series

y_train = pd.DataFrame(y_train.values, columns=['line_total'])
y_val   = pd.DataFrame(y_val.values,   columns=['line_total'])
y_test  = pd.DataFrame(y_test.values,  columns=['line_total'])

# Reset indices for clean CSV output
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print("Ready to save — confirmed shapes:")
print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)

In [ ]:
# ── SAVE y FILES — needed by Baseline and Regression notebooks ───
try:
    y_train.to_csv(at.folder_path / 'y_train.csv', index=False)
    y_val.to_csv(at.folder_path   / 'y_val.csv',   index=False)
    y_test.to_csv(at.folder_path  / 'y_test.csv',  index=False)
    print("All 6 files saved successfully:")
    print(f"  X_train.csv  ({X_train.shape[0]} rows x {X_train.shape[1]} cols)")
    print(f"  X_val.csv    ({X_val.shape[0]} rows x {X_val.shape[1]} cols)")
    print(f"  X_test.csv   ({X_test.shape[0]} rows x {X_test.shape[1]} cols)")
    print(f"  y_train.csv  ({y_train.shape[0]} rows)")
    print(f"  y_val.csv    ({y_val.shape[0]} rows)")
    print(f"  y_test.csv   ({y_test.shape[0]} rows)")
except Exception as e:
    print(e)